# Step 5: The Lakehouse Layer — Delta Lake & ACID

Plain Parquet files have a problem: if a write fails halfway through, or two processes write at the same time, a reader can see a half-written, inconsistent state. There is no transaction.

**Delta Lake** fixes this by adding a transaction log (`_delta_log/`) on top of ordinary Parquet files. Every write — insert, update, delete — becomes one atomic JSON log entry. Readers always see a consistent snapshot, and because old data files are never overwritten, you can time-travel to any previous version.

We use the `deltalake` Python package (built on [delta-rs](https://github.com/delta-io/delta-rs)) — no Spark, no JVM required.

In [ ]:
import os
from pathlib import Path

while not (Path.cwd() / "requirements.txt").exists():
    os.chdir("..")
print("Working directory:", Path.cwd())

## Version 0: an initial write

We aggregate January 2026 into a small daily-revenue-per-region table and write it as a fresh Delta table.

In [ ]:
import shutil

import duckdb
from deltalake import DeltaTable, write_deltalake

DELTA_PATH = "lake/delta/sales_delta"

# Clean slate so this notebook is re-runnable
shutil.rmtree(DELTA_PATH, ignore_errors=True)

con = duckdb.connect()

jan = con.sql("""
    SELECT date, region, SUM(revenue) AS revenue
    FROM 'lake/verkauf/**/*.parquet'
    WHERE jahr = 2026 AND monat = 1
    GROUP BY date, region
""").arrow()

write_deltalake(DELTA_PATH, jan, mode="overwrite")
print(f"Version 0 written: {len(jan)} rows")

## Version 1: an append

Now add February. This is a new transaction — the January data files are untouched.

In [ ]:
feb = con.sql("""
    SELECT date, region, SUM(revenue) AS revenue
    FROM 'lake/verkauf/**/*.parquet'
    WHERE jahr = 2026 AND monat = 2
    GROUP BY date, region
""").arrow()

write_deltalake(DELTA_PATH, feb, mode="append")
print(f"Version 1 written (append): {len(feb)} rows")

## Version 2: an update

Simulate a correction — say, a currency adjustment for Zurich. Plain Parquet has no `UPDATE`; Delta Lake does, and it is transactional: either the whole update lands, or none of it does.

In [ ]:
dt = DeltaTable(DELTA_PATH)
dt.update(predicate="region = 'Zurich'", updates={"revenue": "revenue * 1.1"})
print("Version 2 written (update)")

## Look inside `_delta_log/`

Each write produced exactly one JSON log file. Open the folder in the file explorer, or list it here.

In [ ]:
log_dir = Path(DELTA_PATH) / "_delta_log"
for f in sorted(log_dir.glob("*.json")):
    print(f.name)

In [ ]:
import json

first_log = sorted(log_dir.glob("*.json"))[0]
for line in first_log.read_text().splitlines():
    entry = json.loads(line)
    print(list(entry.keys()))

Each line is one *action*: `add` registers a new Parquet data file as part of the table, `commitInfo` records what operation happened and when, `metaData`/`protocol` describe the schema and format version. A snapshot of the table at any version is just "replay every log entry up to that version and see which files are currently `add`ed".

## Transaction history

`DeltaTable.history()` reads `commitInfo` entries back out as a friendly list — this *is* the audit trail.

In [ ]:
dt = DeltaTable(DELTA_PATH)
for entry in dt.history():
    print(entry.get("version"), "-", entry.get("operation"))

## Time travel

Loading an old version reads the log only up to that point — the update we made afterwards is invisible from version 0's perspective, even though nothing was deleted from disk.

In [ ]:
dt_v0 = DeltaTable(DELTA_PATH, version=0)
v0_total = dt_v0.to_pandas()["revenue"].sum()

dt_latest = DeltaTable(DELTA_PATH)
latest_total = dt_latest.to_pandas()["revenue"].sum()

print(
    "Version 0 (January only, pre-correction) total revenue: "
    f"{v0_total:,.2f}"
)
print(
    "Latest version (Jan+Feb, post-correction) total revenue: "
    f"{latest_total:,.2f}"
)

## Closing the loop

This is the same story as the whole course, compressed into one table:

1. **File format** (Step 1/2, `01`/`02`) — CSV became compressed, columnar Parquet.
2. **The lake** (Step 2/4, `02`/`04`) — files organized into partitions, then moved onto real object storage.
3. **The engine** (Step 3/4, `03`/`04`) — DuckDB queried the same files with schema-on-read, pushdown, and pruning.
4. **The lakehouse layer** (Step 5, `05`, this notebook) — Delta Lake added the one thing plain files never had: transactions.

Each stage solved a problem the previous one made visible. That progression — not any single tool — is the actual lesson.